# Why gross-margin verification no longer skips CONTRACTION claims

Compares analyst Q&A on Aura Q4 2025 (margin contraction language) against audited GM rows.
Baseline commit `dee729f` only reconciled optimistic margin language; see `baseline_dee729f/`.

In [1]:
from pathlib import Path
from src.agents.claim_extractor import ClaimExtractor
from src.agents.numeric_guardrail import NumericGuardrail
from src.db.schema import get_default_db

db = get_default_db(Path("../data/benchmark"))
conn = db.conn
extractor = ClaimExtractor()
guardrail = NumericGuardrail()
claims = extractor.extract_from_db(conn, "AURA", 2025, "Q4")
analyst = next(c for c in claims if c.sentiment_direction == "CONTRACTION")
fc = guardrail.verify_claim(conn, analyst)
print(f"metric={fc.metric} verified={fc.is_verified} audited_gm={fc.audited_value}")
print(fc.explanation)
db.close()

metric=gross_margin verified=True audited_gm=0.61
Audited gross margin compressed from 71.0% to 61.0% (drop of 10.0 percentage points), consistent with the contraction claim.


## What I expected

When evaluating earnings call transcripts, I expected analyst questions about gross margin compression to be classified as negative claims and verified against the audited trend. Instead, the pre-remediation baseline only triggered on expansion or stability keywords, silently dropping valid contraction concerns into generic fallback.


## What the numbers show

The DuckDB quarterly financials show AuraCloud's gross margin fell from 79.0% in Q1 2024 to 61.0% in Q4 2025 (an 1,800 bps total compression, with a 1,000 bps YoY drop in Q4 alone). The analyst's question ('gross margins compressed 1000 bps YoY') was factually correct. Management's reassurance ('stable unit economics') directly contradicted the ledger.


## Decision

We resolved this by routing `CONTRACTION` sentiment through the gross margin verification branch alongside `EXPANSION`. We also introduced proximity-window percentage parsing in `NumericGuardrail` so that explicit numerical claims (e.g., claiming 90% margin when the ledger reports 61%) fail verification on magnitude alone, regardless of qualitative sentiment phrasing.
